## Считаем доли

In [ ]:
import pandas as pd
import numpy as np

df = data.copy()

# суммарный объем по origin + месяц
total = df.groupby(['origin', 'month'])['volume'].transform('sum')

# доля сцепки
df['share'] = df['volume'] / total

## Убираем мусор

In [ ]:
# например: уберем очень маленькие доли
df = df[df['share'] > 0.001]

## Фичи (самое важное место)

In [ ]:
df = df.sort_values(['origin', 'destination', 'cargo_type', 'speed', 'month'])

for lag in [1, 2, 3]:
    df[f'share_lag_{lag}'] = df.groupby(
        ['origin', 'destination', 'cargo_type', 'speed']
    )['share'].shift(lag)

df['share_roll_3'] = df.groupby(
    ['origin', 'destination', 'cargo_type', 'speed']
)['share'].transform(lambda x: x.shift(1).rolling(3).mean())

# месяц как категория
df['month_num'] = df['month'] % 12

df['month_sin'] = np.sin(2 * np.pi * df['month_num'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month_num'] / 12)

origin_total = df.groupby(['origin', 'month'])['volume'].sum().reset_index()
origin_total = origin_total.rename(columns={'volume': 'origin_volume'})

df = df.merge(origin_total, on=['origin', 'month'], how='left')

df = df.dropna()

## Делим на train / val

In [ ]:
train = df[df['month'] > 3]
val = df[df['month'] <= 3]

In [ ]:
from catboost import CatBoostRegressor

cat_features = ['origin', 'destination', 'cargo_type', 'speed']

features = [
    'origin', 'destination', 'cargo_type', 'speed',
    'share_lag_1', 'share_lag_2', 'share_lag_3',
    'share_roll_3',
    'month_sin', 'month_cos',
]

model = CatBoostRegressor(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function='RMSE',
    verbose=100
)

model.fit(
    train[features],
    train['share'],
    cat_features=cat_features,
    eval_set=(val[features], val['share'])
)

## Генерация прогноза

готовим базу сцепок

берем последние доступные:

In [ ]:
latest = df[df['month'] == df['month'].min()]  # последний месяц
base = latest[['origin', 'destination', 'cargo_type', 'speed']].drop_duplicates()

# === лаг 1 ===
lag_1 = df[df['month'] == 1][
    ['origin', 'destination', 'cargo_type', 'speed', 'share']
].rename(columns={'share': 'share_lag_1'})

base = base.merge(
    lag_1,
    on=['origin', 'destination', 'cargo_type', 'speed'],
    how='left'
)


# === лаг 2 ===
lag_2 = df[df['month'] == 2][
    ['origin', 'destination', 'cargo_type', 'speed', 'share']
].rename(columns={'share': 'share_lag_2'})

base = base.merge(
    lag_2,
    on=['origin', 'destination', 'cargo_type', 'speed'],
    how='left'
)


# === лаг 3 ===
lag_3 = df[df['month'] == 3][
    ['origin', 'destination', 'cargo_type', 'speed', 'share']
].rename(columns={'share': 'share_lag_3'})

base = base.merge(
    lag_3,
    on=['origin', 'destination', 'cargo_type', 'speed'],
    how='left'
)

base['share_pred'] = model.predict(base[features])

base['share_pred'] = base.groupby('origin')['share_pred'].transform(
    lambda x: x.clip(lower=0) / x.clip(lower=0).sum()
)

result = base.merge(plan, on='origin', how='left')

result['volume_pred'] = result['share_pred'] * result['volume']